## Import Required Libraries

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv("../.env")

print(" All imports done!")

d:\ask-my-docs\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 All imports done!


## Loading the Document

In [15]:
import pdfplumber

text = ""
with pdfplumber.open("../data/data document.pdf") as pdf:
    for page in pdf.pages:
        extracted = page.extract_text()
        if extracted:
            cleaned = " ".join(extracted.split())
            text += cleaned + "\n"

print(f"Total characters: {len(text)}")
print("\nFirst 500 characters:")
print(text[:500])

Total characters: 2721

First 500 characters:
Fundamentals of Image Processing 1. Color Image Processing Color image processing deals with analyzing and manipulating images that contain color information. Unlike grayscale images (single intensity channel), color images typically use multiple channels such as RGB (Red, Green, Blue). Key Concepts: ● Color Models: ○ RGB (used in displays) ○ HSV (Hue, Saturation, Value – useful for segmentation) ○ CMY/CMYK (printing) ● Applications: ○ Object detection ○ Image segmentation ○ Medical imaging ○ Co


## Breaking into Chunks

In [16]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)
print(f"Total chunks:{len(chunks)}")

Total chunks:7


## Store in ChromaDB

In [19]:
client_db = chromadb.PersistentClient(path="../data/chromadb_rag")

try:
    client_db.delete_collection("my_pdf_docs")
except:
    pass

collection = client_db.get_or_create_collection(name="my_pdf_docs")

collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"Total chunks stored: {collection.count()}")

Total chunks stored: 7


## Retrieval

In [20]:
query = " What are morphological Operations ? "

results = collection.query(
    query_texts=[query],
    n_results=3
)

retrieved_chunks = results['documents'][0]

print("Retrived chunks:")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\nChunk {i+1}: \n{chunk}")

Retrived chunks:

Chunk 1: 
4. Morphological Operations Morphological operations are used to process images based on shapes. They are mainly applied on binary images. Basic Operations: (a) Erosion ● Removes pixels from object boundaries ● Shrinks objects (b) Dilation ● Adds pixels to boundaries ● Expands objects (c) Opening ● Erosion followed by dilation ● Removes small objects/noise (d) Closing ● Dilation followed by erosion ● Fills small holes Applications: ● Noise removal ● Shape extraction ● Object detection 5. Edge

Chunk 2: 
○ Image segmentation ○ Medical imaging ○ Computer vision systems Advantages: ● Provides more information than grayscale images ● Helps in better feature extraction and recognition 2. Image Enhancement Image enhancement improves the visual quality of an image or highlights important features. Types: (a) Smoothing (Blurring) Used to reduce noise and minor details. Techniques: ● Mean Filter ● Gaussian Filter ● Median Filter Use Cases: ● Preprocessing before edge

## Retrieving from LLM

In [21]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using ONLY the context below.
If answer is not in context, say "I don't know".
At the end mention which part of context you used.

Context:
{context}

Question: {query}
"""

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("Answer:")
print(response.choices[0].message.content)


Answer:
Morphological operations are used to process images based on shapes, mainly applied on binary images, and include basic operations such as erosion, dilation, opening, and closing.

I used: Part 4. Morphological Operations of the context.
